# 25 - Apply Fine-Tashkeel to Multiple Models

**Goal**: Apply Fine-Tashkeel to improve existing predictions from multiple models

**Models to Process**:
- seamless_m4t (best ASR)
- artst_asr
- tartel_whisper
- whisper_quran_lora
- shakkelha (best tashkeel)
- shakkala
- catt
- fine_tashkeel
- byt5_glonor
- mushkil

**Approach**: Load existing predictions, apply Fine-Tashkeel to add/improve diacritics, evaluate, create submissions

In [1]:
# Install dependencies
!pip install -q transformers torch accelerate tqdm sentencepiece pyarabic


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Setup
import os, sys, json, re, torch, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Paths
DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [ ]:
# Load Fine-Tashkeel
print("Loading Fine-Tashkeel...")
TASHKEEL_MODEL = 'basharalrfooh/Fine-Tashkeel'
tashkeel_tokenizer = AutoTokenizer.from_pretrained(TASHKEEL_MODEL)
tashkeel_model = AutoModelForSeq2SeqLM.from_pretrained(
    TASHKEEL_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
tashkeel_model.eval()
print(f"Fine-Tashkeel loaded: {sum(p.numel() for p in tashkeel_model.parameters()):,} params")

Loading Fine-Tashkeel...


`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
# Load data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

print(f"Dev: {len(dev_input)}, Test: {len(test_input)}")

In [ ]:
# Define models to process
MODELS = [
    'seamless_m4t',
    'artst_asr',
    'tarteel_whisper',
    'whisper_quran_lora',
    'shakkelha',
    'shakkala',
    'catt',
    'fine_tashkeel',
    'byt5_glonor',
    'mushkil',
]

print(f"Will process {len(MODELS)} models")

In [ ]:
# Diacritization function
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')

def remove_diacritics(text):
    """Remove all diacritics from text."""
    return DIACRITIC_PATTERN.sub('', text)

@torch.inference_mode()
def diacritize_text(text):
    """Add diacritics to undiacritized Arabic text."""
    if not text or not text.strip():
        return text
    
    clean_text = remove_diacritics(text)
    
    inputs = tashkeel_tokenizer(clean_text, return_tensors="pt", padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    outputs = tashkeel_model.generate(
        **inputs,
        max_length=512,
        num_beams=4,
        early_stopping=True
    )
    
    result = tashkeel_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.strip()

In [ ]:
# Metrics - Diac-style evaluation
from diac_evaluation import compute_metrics, print_metrics_table

print("Diac-style evaluation loaded!")

## Process All Models

In [ ]:
# Process each model
results = []

for model_key in MODELS:
    print(f"\n{'='*60}")
    print(f"Processing: {model_key}")
    print(f"{'='*60}")
    
    # Check if dev predictions exist
    dev_file = OUTPUT_DIR / f'{model_key}_dev_predictions.json'
    test_file = OUTPUT_DIR / f'{model_key}_test_predictions.json'
    
    if not dev_file.exists():
        print(f"  Skipping {model_key} - no dev predictions found")
        continue
    
    # Load predictions
    with open(dev_file, 'r', encoding='utf-8') as f:
        dev_preds = json.load(f)
    dev_lookup = {p['id']: p['text_diacritized'] for p in dev_preds}
    print(f"  Loaded {len(dev_lookup)} dev predictions")
    
    # Apply Fine-Tashkeel
    enhanced_preds = []
    for item in tqdm(dev_input, desc=f"  Enhancing {model_key}"):
        if item['id'] in dev_lookup:
            original = dev_lookup[item['id']]
            undiac = remove_diacritics(original)
            try:
                enhanced = diacritize_text(undiac)
            except:
                enhanced = original
        else:
            enhanced = item['text_undiacritized']
        
        enhanced_preds.append({'id': item['id'], 'text_diacritized': enhanced})
    
    # Evaluate
    metrics = compute_metrics(enhanced_preds, dev_output)
    print_metrics_table(metrics, f"{model_key} - Fine-Tashkeel Enhanced")
    
    # Save enhanced predictions
    enhanced_dev_file = OUTPUT_DIR / f'{model_key}_ft_dev_predictions.json'
    with open(enhanced_dev_file, 'w', encoding='utf-8') as f:
        json.dump(enhanced_preds, f, ensure_ascii=False, indent=2)
    
    results.append({
        'model': model_key,
        'der': metrics['DER_case_yes_nodiac_yes'],
        'wer': metrics['WER_case_yes_nodiac_yes'],
        'ser': metrics['SER_case_yes_nodiac_yes'],
    })
    
    # Process test set if available
    if test_file.exists():
        with open(test_file, 'r', encoding='utf-8') as f:
            test_preds = json.load(f)
        test_lookup = {p['id']: p['text_diacritized'] for p in test_preds}
        print(f"  Processing {len(test_lookup)} test predictions...")
        
        enhanced_test = []
        for item in tqdm(test_input, desc=f"  Test {model_key}"):
            if item['id'] in test_lookup:
                original = test_lookup[item['id']]
                undiac = remove_diacritics(original)
                try:
                    enhanced = diacritize_text(undiac)
                except:
                    enhanced = original
            else:
                enhanced = item['text_undiacritized']
            
            enhanced_test.append({'id': item['id'], 'text_diacritized': enhanced})
        
        # Save test predictions
        enhanced_test_file = OUTPUT_DIR / f'{model_key}_ft_test_predictions.json'
        with open(enhanced_test_file, 'w', encoding='utf-8') as f:
            json.dump(enhanced_test, f, ensure_ascii=False, indent=2)
        
        # Create submission
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        json_file = SUBMISSION_DIR / f'{model_key}_ft_submission.json'
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(enhanced_test, f, ensure_ascii=False, indent=2)
        
        zip_file = SUBMISSION_DIR / f'{model_key}_ft_submission_{timestamp}.zip'
        with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.write(json_file, 'submission.json')
        
        print(f"  Submission: {zip_file}")

In [ ]:
# Summary
print("\n" + "="*80)
print("SUMMARY - Fine-Tashkeel Enhanced Models")
print("="*80)
print(f"{'Model':<25} {'DER':>10} {'WER':>10} {'SER':>10}")
print("-"*80)

for r in sorted(results, key=lambda x: x['wer']):
    print(f"{r['model']:<25} {r['der']:>9.2f}% {r['wer']:>9.2f}% {r['ser']:>9.2f}%")

print("="*80)
print("\nNote: Metrics are computed with Case Ending=Yes, No-Diacritic=Include")
print("="*80)

## Compare Original vs Enhanced

In [ ]:
# Load original results for comparison
import pandas as pd

# Original scores from evaluation
original_scores = {
    'seamless_m4t': (42.82, 65.94, 85.06),
    'artst_asr': (45.89, 82.39, 100.00),
    'tarteel_whisper': (84.93, 85.08, 100.00),
    'whisper_quran_lora': (97.36, 99.98, 100.00),
    'shakkelha': (13.14, 39.47, 83.84),
    'shakkala': (19.70, 56.37, 96.95),
    'catt': (28.34, 48.17, 100.00),
    'fine_tashkeel': (100.00, 100.00, 100.00),  # Check this
    'byt5_glonor': (46.96, 69.64, 100.00),
    'mushkil': (74.17, 97.68, 99.70),
}

print(f"{'Model':<25} {'Orig DER':>10} {'FT DER':>10} {'Orig WER':>10} {'FT WER':>10}")
print("-"*70)

for r in results:
    model = r['model']
    if model in original_scores:
        orig_der, orig_wer, _ = original_scores[model]
        print(f"{model:<25} {orig_der:>9.2f}% {r['der']:>9.2f}% {orig_wer:>9.2f}% {r['wer']:>9.2f}%")
    else:
        print(f"{model:<25} {'N/A':>9} {r['der']:>9.2f}% {'N/A':>9} {r['wer']:>9.2f}%")